# 🚀 ENTRENAMIENTO LLAMA-3.2-1B V5 CON EARLY STOPPING

## 📊 Objetivo
Entrenar Llama-3.2-1B con dataset híbrido (351 ejemplos) y early stopping

## ✨ Novedades V5:
- ✅ Dataset híbrido (tonalidad + RAG genérico + formato real)
- ✅ Early stopping (para en loss óptimo)
- ✅ TensorBoard para monitoreo
- ✅ Sin errores tipográficos esperados

---

## 📦 PASO 1: INSTALAR DEPENDENCIAS

In [ ]:
%%capture
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece einops
print("✅ Dependencias instaladas")

## 🎮 PASO 2: VERIFICAR GPU

In [ ]:
import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Activa GPU: Runtime → Change runtime type → T4 GPU")

print("=" * 70)

## 📤 PASO 3: SUBIR DATASET HÍBRIDO

**Sube el archivo:** `dataset_pedagogico_hibrido_500.json` (351 ejemplos)

In [ ]:
from google.colab import files
import json

print("📤 Sube dataset_pedagogico_hibrido_500.json")
uploaded = files.upload()

dataset_file = 'dataset_pedagogico_hibrido_500.json'

if dataset_file in uploaded:
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n✅ Dataset: {len(data)} ejemplos")
    print(f"\n📊 Distribución esperada:")
    print(f"   - Tonalidad pedagógica: ~140 ejemplos")
    print(f"   - RAG genérico: ~135 ejemplos")
    print(f"   - RAG formato real: ~76 ejemplos")
else:
    print("❌ Error: dataset no encontrado")

## 🔑 PASO 4: CONFIGURAR TOKEN HUGGINGFACE

In [ ]:
from getpass import getpass
HF_TOKEN = getpass("Token HuggingFace: ")
print("✅ Token configurado")

## 📊 PASO 5: ACTIVAR TENSORBOARD

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 🎯 PASO 6: CONFIGURACIÓN DEL MODELO V5

In [ ]:
from peft import TaskType

MODEL_CONFIG = {
    'display_name': 'Llama-3.2-1B-V5-Hybrid',
    'model_id': 'meta-llama/Llama-3.2-1B-Instruct',
    'epochs': 40,  # Más épocas, pero early stopping parará antes
    'batch_size': 4,
    'grad_accum': 2,
    'learning_rate': 5e-5,  # Ligeramente más alto que V4
    'warmup_steps': 50,
    'early_stopping_patience': 3,  # ✅ NUEVO: Para si no mejora en 3 épocas
    'early_stopping_threshold': 0.01,  # ✅ NUEVO: Mejora mínima requerida
    'time_estimate': '50-60 min',
    'lora_config': {
        'r': 16,  # ✅ Más capacidad que V4 (era 8)
        'lora_alpha': 32,  # ✅ Más capacidad que V4 (era 16)
        'target_modules': [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj'
        ],
        'lora_dropout': 0.1,
        'bias': 'none',
        'task_type': TaskType.CAUSAL_LM
    }
}

print("✅ Configuración V5 cargada")
print(f"\n📊 Parámetros clave:")
print(f"   - Épocas máximas: {MODEL_CONFIG['epochs']}")
print(f"   - Early stopping patience: {MODEL_CONFIG['early_stopping_patience']}")
print(f"   - Learning rate: {MODEL_CONFIG['learning_rate']}")
print(f"   - LoRA rank: {MODEL_CONFIG['lora_config']['r']}")
print(f"   - LoRA alpha: {MODEL_CONFIG['lora_config']['lora_alpha']}")

## 🏋️ PASO 7: PREPARAR DATASET

In [ ]:
from datasets import Dataset

print("📚 Preparando dataset...")

with open(dataset_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    """Formato especial para Llama-3.2-1B"""
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return {"text": text}

dataset = Dataset.from_list(data).map(format_instruction)
print(f"✅ {len(dataset)} ejemplos formateados")

## 🤖 PASO 8: CARGAR MODELO Y TOKENIZER

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

print("🤖 Cargando modelo y tokenizer...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG['model_id'],
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Modelo con cuantización 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG['model_id'],
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

print("✅ Modelo y tokenizer cargados")

## 🔧 PASO 9: APLICAR LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

print("🔧 Aplicando LoRA...")

lora_config = LoraConfig(**MODEL_CONFIG['lora_config'])
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"✅ LoRA aplicado")
print(f"📊 Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"📊 Parámetros totales: {total:,}")

## 📝 PASO 10: TOKENIZAR DATASET

In [ ]:
from transformers import DataCollatorForLanguageModeling

print("📝 Tokenizando dataset...")

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("✅ Dataset tokenizado")

## 🚀 PASO 11: ENTRENAR CON EARLY STOPPING

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

print("=" * 70)
print(f"🚀 INICIANDO ENTRENAMIENTO - {MODEL_CONFIG['display_name']}")
print("=" * 70)
print(f"⏱️  Tiempo estimado: {MODEL_CONFIG['time_estimate']}")
print(f"📊 TensorBoard: logs/llama3_v5")
print(f"🛑 Early stopping: {MODEL_CONFIG['early_stopping_patience']} épocas sin mejora")
print()

training_args = TrainingArguments(
    output_dir="./lora_model_llama3_v5",
    num_train_epochs=MODEL_CONFIG['epochs'],
    per_device_train_batch_size=MODEL_CONFIG['batch_size'],
    gradient_accumulation_steps=MODEL_CONFIG['grad_accum'],
    learning_rate=MODEL_CONFIG['learning_rate'],
    fp16=True,
    logging_dir="./logs/llama3_v5",
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    warmup_steps=MODEL_CONFIG['warmup_steps'],
    weight_decay=0.01,
    max_grad_norm=1.0,
    report_to="tensorboard",
    optim="paged_adamw_8bit",
    disable_tqdm=False,
    logging_first_step=True,
    load_best_model_at_end=True,  # ✅ NUEVO: Cargar mejor modelo al final
    metric_for_best_model="loss",  # ✅ NUEVO: Métrica para early stopping
    greater_is_better=False,  # ✅ NUEVO: Loss menor es mejor
    evaluation_strategy="epoch",  # ✅ NUEVO: Evaluar cada época
)

# ✅ NUEVO: Callback de early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=MODEL_CONFIG['early_stopping_patience'],
    early_stopping_threshold=MODEL_CONFIG['early_stopping_threshold']
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # ✅ NUEVO: Usar mismo dataset para eval
    data_collator=data_collator,
    callbacks=[early_stopping]  # ✅ NUEVO: Agregar early stopping
)

# ENTRENAR
trainer.train()

print("\n" + "=" * 70)
print(f"✅ ENTRENAMIENTO COMPLETADO - {MODEL_CONFIG['display_name']}")
print("=" * 70)

# Buscar el último loss
final_loss = None
for log_entry in reversed(trainer.state.log_history):
    if 'loss' in log_entry:
        final_loss = log_entry['loss']
        break

if final_loss:
    print(f"\n📊 Loss final: {final_loss:.4f}")
    print(f"📊 Épocas completadas: {trainer.state.epoch:.0f}")
    
    if final_loss < 0.15:
        print("\n✅ ¡EXCELENTE! Loss < 0.15 - Modelo óptimo")
    elif final_loss < 0.25:
        print("\n✅ ¡MUY BUENO! Loss < 0.25 - Modelo de calidad")
    else:
        print("\n⚠️  Loss > 0.25 - Considera más ejemplos o ajustar hiperparámetros")
else:
    print("\n⚠️  No se pudo obtener el loss final")

## 💾 PASO 12: GUARDAR ADAPTADORES

In [ ]:
print("💾 Guardando adaptadores LoRA...")

output_dir = "./lora_adapters_llama3_v5"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Adaptadores guardados en: {output_dir}")
print("\n📋 Archivos guardados:")
print("   - adapter_config.json")
print("   - adapter_model.safetensors")
print("   - tokenizer files")

## 🧪 PASO 13: PROBAR MODELO

In [ ]:
print("=" * 70)
print("🧪 PROBANDO MODELO ENTRENADO")
print("=" * 70)

model.eval()

# Prompt de prueba 1: Sin contexto (tonalidad)
test_prompt_1 = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explica qué es una derivada

Necesito entender el concepto de derivada para mi curso de cálculo<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Prompt de prueba 2: Con contexto RAG (formato real)
test_prompt_2 = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde ÚNICAMENTE basándote en el CONTEXTO proporcionado.<|eot_id|><|start_header_id|>user<|end_header_id|>

CONTEXTO: UNIDAD 1: FUNDAMENTOS DE DATA SCIENCE Y PYTHON

PREGUNTA: ¿Cuál es la unidad 1?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

def test_generation(prompt, test_name):
    print(f"\n{'='*70}")
    print(f"🧪 {test_name}")
    print(f"{'='*70}")
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    
    print(f"\n📝 RESPUESTA:")
    print("-" * 70)
    print(response[:500] if len(response) > 500 else response)
    print("-" * 70)

# Ejecutar pruebas
test_generation(test_prompt_1, "PRUEBA 1: Tonalidad pedagógica (sin contexto)")
test_generation(test_prompt_2, "PRUEBA 2: RAG con formato real (con contexto)")

print("\n" + "=" * 70)
print("✅ PRUEBAS COMPLETADAS")
print("=" * 70)
print("\n🔍 VERIFICAR:")
print("   ✅ Tonalidad pedagógica (emojis, tono amigable)")
print("   ✅ Sin errores tipográficos (182¹, independência, Perù)")
print("   ✅ Respuesta precisa del contexto")
print("   ✅ Sin inventar información")

## 📥 PASO 14: DESCARGAR ADAPTADORES

In [ ]:
import shutil
from google.colab import files
import os

print("📦 Comprimiendo adaptadores...")

zip_name = "lora_adapters_llama3_v5"
shutil.make_archive(zip_name, 'zip', output_dir)
print(f"✅ {zip_name}.zip creado")

print("\n📥 Descargando archivo...")
files.download(f"{zip_name}.zip")

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime lora_adapters_llama3_v5.zip")
print("   2. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   3. Actualiza lora_integration.py:")
print("      adapters_path = './fine_tuning/lora_adapters/lora_adapters_llama3_v5'")
print("   4. Reinicia backend: python main.py")
print("   5. Prueba en la app con tus sílabos reales")
print("\n🎉 ¡Listo para producción!")

---

## 📊 RESUMEN DEL MODELO V5

### **Mejoras vs V4:**
- ✅ Dataset híbrido (351 ejemplos vs 79)
- ✅ Early stopping (para en loss óptimo)
- ✅ LoRA rank 16 (vs 8 en V4)
- ✅ Aprende formato real de sílabos
- ✅ Sin post-procesamiento necesario

### **Resultado esperado:**
- Loss: 0.08 - 0.12
- Sin errores tipográficos
- Respuestas precisas del contexto RAG
- Escalable a nuevos sílabos

---